In [85]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


In [86]:
from google.colab import drive
drive.mount('/content/drive')

print("Drive mounted successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully!


In [87]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Path to your folder in Drive
PATH = '/content/drive/MyDrive/olist_project/Dataset/'

print("Path set! Loading files now...")

Path set! Loading files now...


Load all datasets:

In [88]:
orders      = pd.read_csv(PATH + 'olist_orders_dataset.csv')
customers   = pd.read_csv(PATH + 'olist_customers_dataset.csv')
order_items = pd.read_csv(PATH + 'olist_order_items_dataset.csv')
payments    = pd.read_csv(PATH + 'olist_order_payments_dataset.csv')
reviews     = pd.read_csv(PATH + 'olist_order_reviews_dataset.csv')
products    = pd.read_csv(PATH + 'olist_products_dataset.csv')
sellers     = pd.read_csv(PATH + 'olist_sellers_dataset.csv')
category    = pd.read_csv(PATH + 'product_category_name_translation.csv')

print("All 8 files loaded!")
print(f"Orders:     {orders.shape}")
print(f"Customers:  {customers.shape}")
print(f"Items:      {order_items.shape}")
print(f"Payments:   {payments.shape}")
print(f"Reviews:    {reviews.shape}")
print(f"Products:   {products.shape}")
print(f"Sellers:    {sellers.shape}")
print(f"Category:   {category.shape}")

All 8 files loaded!
Orders:     (99441, 8)
Customers:  (99441, 5)
Items:      (112650, 7)
Payments:   (103886, 5)
Reviews:    (99224, 7)
Products:   (32951, 9)
Sellers:    (3095, 4)
Category:   (71, 2)


Data cleaning + merging into master table


In [89]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Date columns fixed!")
print(orders.dtypes)

Date columns fixed!
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


Filter only delivered orders

We only want completed orders for analysis. Remove cancelled, unavailable orders

In [90]:
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f"Total orders: {len(orders)}")
print(f"Delivered orders: {len(orders_delivered)}")
print(f"Removed: {len(orders) - len(orders_delivered)} non-delivered orders")

Total orders: 99441
Delivered orders: 96478
Removed: 2963 non-delivered orders


 Calculate delivery delay

This is a key feature — how many days late or early was each delivery? Negative = early, Positive = late.

In [91]:
orders_delivered['delivery_delay_days'] = (
    orders_delivered['order_delivered_customer_date'] -
    orders_delivered['order_estimated_delivery_date']
).dt.days

orders_delivered['is_late'] = (orders_delivered['delivery_delay_days'] > 0).astype(int)

print("Delay stats:")
print(orders_delivered['delivery_delay_days'].describe())
print(f"\nLate deliveries: {orders_delivered['is_late'].sum()} ({orders_delivered['is_late'].mean()*100:.1f}%)")

Delay stats:
count    96470.000000
mean       -11.875889
std         10.182105
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

Late deliveries: 6534 (6.8%)


Merge all 8 tables into one master

Join everything step by step. Think of this like SQL JOINs — we connect tables using common keys.

In [92]:
df = orders_delivered.merge(customers, on='customer_id', how='left')
df = df.merge(order_items, on='order_id', how='left')
df = df.merge(payments.groupby('order_id')['payment_value'].sum().reset_index(), on='order_id', how='left')
df = df.merge(reviews[['order_id','review_score']], on='order_id', how='left')
df = df.merge(products, on='product_id', how='left')
df = df.merge(category, on='product_category_name', how='left')
df = df.merge(sellers, on='seller_id', how='left')

print(f"Master table shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Master table shape: (110840, 34)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'is_late', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_value', 'review_score', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


Clean the master table

Drop columns we don't need, rename for clarity, fill nulls.

In [93]:
df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')
df['review_score'] = df['review_score'].fillna(df['review_score'].median())

df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)
df['order_year']  = df['order_purchase_timestamp'].dt.year

print("Nulls remaining:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nFinal master table: {df.shape[0]} rows, {df.shape[1]} columns")

Nulls remaining:
order_approved_at                  15
order_delivered_carrier_date        2
order_delivered_customer_date       8
delivery_delay_days                 8
payment_value                       3
product_category_name            1545
product_name_lenght              1545
product_description_lenght       1545
product_photos_qty               1545
product_weight_g                   18
product_length_cm                  18
product_height_cm                  18
product_width_cm                   18
dtype: int64

Final master table: 110840 rows, 36 columns


 Save master table to Drive

Save the cleaned data so every future day loads this directly — no need to redo cleaning.



In [94]:
df.to_csv('/content/drive/MyDrive/olist_project/master_table.csv', index=False)
print("master_table.csv saved to Drive!")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")

master_table.csv saved to Drive!
Rows: 110840 | Columns: 36
